# Appendix D Companion Notebook: NoSQL and Document-Oriented Databases

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/appendices/Appendix_D_NoSQL_and_Document_Oriented_Databases.ipynb)

This notebook accompanies Appendix D of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




This notebook turns Appendix D into guided practice with semi-structured business records. A synthetic e-commerce case connects JSON-like documents, MongoDB collections, document design, CRUD operations, nested queries, indexes, aggregation pipelines, pandas integration, data-quality checks, governance, and a customer-feedback mini-case.

## How to use this notebook

Run the cells from top to bottom. Read the explanation before each code block and inspect both the documents and the analytical grain of each result.

The required exercises use `mongomock`, an in-memory implementation of much of the PyMongo interface. It lets the notebook run in Google Colab without installing or configuring a MongoDB server. The query syntax closely follows PyMongo, but the mock does not reproduce production networking, authentication, distributed scaling, query-planner behavior, transactions, or performance. An optional section shows how to test a protected MongoDB Atlas connection without placing a password in the notebook.

Before sharing the notebook, restart the runtime and run all cells. A clean run confirms that the collections, variables, and saved outputs do not depend on hidden state or an accidental execution order.

## Why this matters (business framing)

Business data are not always born as rectangular tables. Product catalogs contain category-specific attributes, reviews include optional metadata and free text, and digital events often arrive as nested JSON records. A document-oriented database can preserve this context while supporting focused retrieval, aggregation, and transfer to Python.

The business value does not come from choosing NoSQL because it is newer. It comes from matching the storage design to the record structure, the expected access pattern, the governance requirements, and the decision that the analysis must support.

## Agenda

1. Setup and reproducibility
2. NoSQL families and database choice
3. Relational rows versus nested documents
4. A synthetic e-commerce document database
5. CRUD operations and nested queries
6. Embedding versus referencing
7. Indexes and access patterns
8. Aggregation pipelines
9. MongoDB-to-pandas workflow
10. Data quality, schema validation, and integrity checks
11. Optional MongoDB Atlas connection
12. Customer feedback and purchase analytics mini-case
13. Governance, handoff, and exercises

## Learning objectives

After completing this notebook, you should be able to distinguish major NoSQL data models, explain the role of a database, collection, and document, represent nested and optional fields, choose between embedding and referencing, perform basic create, read, update, and delete operations, filter nested fields and arrays, define indexes from access patterns, construct aggregation pipelines with `$match`, `$project`, `$group`, `$unwind`, `$sort`, and `$lookup`, convert focused query results into pandas DataFrames, detect inconsistent fields and data types, validate a document contract, audit referenced identifiers, protect connection credentials, and assemble a reproducible product-level decision dataset.

## Connection map

Appendix A introduced Google Colab as the workspace. Appendix B introduced Python, pandas, visualization, and reproducible file handling. Appendix C introduced relational schemas and SQL. Appendix D adds document-oriented storage for records whose fields are nested, optional, or likely to evolve. Later text-mining and machine-learning chapters assume that these records have been filtered, validated, and converted to an appropriate analytical grain before modeling.

In [ ]:
# ============================================================
# 1. Setup and reproducibility
# ============================================================
from pathlib import Path
from datetime import datetime, timedelta, timezone
from copy import deepcopy
import hashlib
import importlib.util
import json
import os
import platform
import random
import shutil
import subprocess
import sys

# Install the two MongoDB-related packages only when they are absent.
required_packages = {
    'pymongo': 'pymongo[srv]',
    'mongomock': 'mongomock',
}
missing_packages = [
    pip_name
    for module_name, pip_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', *missing_packages
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from bson import BSON, ObjectId
import mongomock
from pymongo import ASCENDING, DESCENDING
from pymongo import MongoClient as PyMongoClient
from jsonschema import Draft202012Validator, FormatChecker

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

WORK_DIR = Path('appendix_d_workspace')
DATA_DIR = WORK_DIR / 'data'
OUT_DIR = WORK_DIR / 'outputs'

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 90)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

print(f'Python: {platform.python_version()}')
print(f'pandas: {pd.__version__}')
print(f'mongomock: {mongomock.__version__}')
print(f'Working directory: {Path.cwd().resolve()}')
print('Execution mode: in-memory MongoDB-compatible learning environment')

## Utility functions

These helpers keep the examples readable. They display compact tables, convert nested documents into JSON-safe objects, move query results into pandas, profile document fields, measure BSON document size, save reproducible artifacts, and calculate checksums for the final handoff record.

In [ ]:
# ============================================================
# Utility functions
# ============================================================
def show_table(frame, rows=10):
    """Display a compact copy of a pandas DataFrame."""
    display(frame.head(rows).copy())


def json_safe(value):
    """Recursively convert BSON, datetime, pandas, and NumPy values for JSON."""
    if isinstance(value, (datetime, pd.Timestamp)):
        timestamp = pd.Timestamp(value)
        if timestamp.tzinfo is None:
            timestamp = timestamp.tz_localize('UTC')
        else:
            timestamp = timestamp.tz_convert('UTC')
        return timestamp.isoformat().replace('+00:00', 'Z')
    if isinstance(value, ObjectId):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    if pd.isna(value) if not isinstance(value, (list, dict, tuple, set)) else False:
        return None
    return value


def print_document(document):
    """Pretty-print one document without exposing non-JSON Python types."""
    print(json.dumps(json_safe(document), indent=2, ensure_ascii=False))


def cursor_df(cursor):
    """Convert a PyMongo-style cursor or list of documents into a flat DataFrame."""
    records = [json_safe(document) for document in list(cursor)]
    return pd.json_normalize(records) if records else pd.DataFrame()


def save_json(payload, path):
    """Save a JSON-serializable object with readable indentation."""
    with open(path, 'w', encoding='utf-8') as handle:
        json.dump(json_safe(payload), handle, indent=2, ensure_ascii=False)


def save_jsonl(documents, path):
    """Save one JSON object per line."""
    with open(path, 'w', encoding='utf-8') as handle:
        for document in documents:
            handle.write(json.dumps(json_safe(document), ensure_ascii=False) + '\n')


def sha256_file(path):
    """Return a SHA-256 checksum for a file."""
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(65536), b''):
            digest.update(block)
    return digest.hexdigest()


def save_current_figure(filename):
    """Finish, save, display, and close the current Matplotlib figure."""
    path = OUT_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    return path


def bson_size_bytes(document):
    """Return the encoded BSON size of one document."""
    return len(BSON.encode(deepcopy(document)))


def iter_field_paths(document, prefix=''):
    """Yield flattened field paths while keeping arrays as one field type."""
    for key, value in document.items():
        path = f'{prefix}.{key}' if prefix else str(key)
        yield path, value
        if isinstance(value, dict):
            yield from iter_field_paths(value, prefix=path)


def field_profile(documents):
    """Summarize field coverage and observed Python types across documents."""
    documents = list(documents)
    total = len(documents)
    profile = {}
    for document in documents:
        seen = set()
        for path, value in iter_field_paths(document):
            record = profile.setdefault(path, {'non_null_count': 0, 'types': set()})
            if value is not None and path not in seen:
                record['non_null_count'] += 1
            record['types'].add(type(value).__name__)
            seen.add(path)
    rows = []
    for path, record in profile.items():
        rows.append({
            'field_path': path,
            'non_null_count': record['non_null_count'],
            'coverage': record['non_null_count'] / total if total else np.nan,
            'observed_types': ', '.join(sorted(record['types'])),
        })
    return pd.DataFrame(rows).sort_values(['coverage', 'field_path'], ascending=[True, True]).reset_index(drop=True)


def dataframe_records(frame):
    """Convert DataFrame rows into MongoDB-safe Python dictionaries."""
    records = []
    for record in frame.to_dict(orient='records'):
        cleaned = {}
        for key, value in record.items():
            if isinstance(value, np.generic):
                value = value.item()
            if isinstance(value, pd.Timestamp):
                value = value.to_pydatetime()
            if isinstance(value, float) and np.isnan(value):
                value = None
            cleaned[key] = value
        records.append(cleaned)
    return records

## D.1 NoSQL in the business analytics workflow

NoSQL is best read as *Not Only SQL*. The category includes several storage models, and each model organizes information around a different type of access. The first task is therefore not to select a product name. It is to identify the structure of the record and the question that must be answered.

In [ ]:
# ============================================================
# D.1.1 Major NoSQL families and business uses
# ============================================================
nosql_models = pd.DataFrame([
    {
        'data_model': 'Key-value',
        'basic_idea': 'One key retrieves one value',
        'common_business_use': 'Sessions, carts, preferences, and caching',
        'example_systems': 'Redis, Riak, Amazon DynamoDB',
    },
    {
        'data_model': 'Document',
        'basic_idea': 'JSON-like records with nested fields and arrays',
        'common_business_use': 'Reviews, product catalogs, events, and survey responses',
        'example_systems': 'MongoDB, CouchDB, ArangoDB',
    },
    {
        'data_model': 'Column-family',
        'basic_idea': 'Distributed records organized around column families',
        'common_business_use': 'High-volume event streams and sensor histories',
        'example_systems': 'Apache Cassandra, HBase, ScyllaDB',
    },
    {
        'data_model': 'Graph',
        'basic_idea': 'Entities are nodes and relationships are edges',
        'common_business_use': 'Fraud rings, referrals, networks, and recommendation paths',
        'example_systems': 'Neo4j, Amazon Neptune, TigerGraph',
    },
])

business_choice_examples = pd.DataFrame([
    {'business_task': 'Invoice and payment reporting', 'natural_start': 'Relational database', 'reason': 'Stable fields, integrity rules, and repeatable joins'},
    {'business_task': 'Flexible product attributes', 'natural_start': 'Document database', 'reason': 'Different categories need different nested attributes'},
    {'business_task': 'Fast retrieval of an active shopping cart', 'natural_start': 'Key-value store', 'reason': 'A known key should return the current state quickly'},
    {'business_task': 'Tracing connected accounts and devices', 'natural_start': 'Graph database', 'reason': 'Relationships are the central analytical object'},
])

show_table(nosql_models, rows=10)
show_table(business_choice_examples, rows=10)

## D.2 From relational rows to documents

A relational design separates entities into tables and restores context through keys and joins. A document design can keep related context in one nested record. Neither design is universally superior. The correct comparison is between the expected business queries, update patterns, integrity needs, and governance requirements.

In [ ]:
# ============================================================
# D.2.1 A nested customer-review document
# ============================================================
sample_review_document = {
    '_id': 'R1007',
    'customer': {
        'customer_id': 'C501',
        'loyalty_level': 'Gold',
    },
    'product': {
        'product_id': 'P204',
        'name': 'Wireless Earbuds',
        'category': 'Electronics',
    },
    'rating': 4,
    'review_text': 'Fast shipping and clear sound.',
    'aspects': ['shipping', 'sound'],
    'source': 'mobile_app',
    'created_at': datetime(2026, 3, 14, 10, 15),
}

print_document(sample_review_document)

sample_flat = pd.json_normalize([json_safe(sample_review_document)])
show_table(sample_flat)

In [ ]:
# ============================================================
# D.2.2 Restoring the same context from relational tables
# ============================================================
customers_table = pd.DataFrame([
    {'customer_id': 'C501', 'loyalty_level': 'Gold'},
])
products_table = pd.DataFrame([
    {'product_id': 'P204', 'name': 'Wireless Earbuds', 'category': 'Electronics'},
])
reviews_table = pd.DataFrame([
    {
        'review_id': 'R1007',
        'customer_id': 'C501',
        'product_id': 'P204',
        'rating': 4,
        'review_text': 'Fast shipping and clear sound.',
        'source': 'mobile_app',
        'created_at': '2026-03-14T10:15:00Z',
    },
])

joined_review = (
    reviews_table
    .merge(customers_table, on='customer_id', how='left', validate='many_to_one')
    .merge(products_table, on='product_id', how='left', validate='many_to_one')
)

comparison = pd.DataFrame([
    {
        'design': 'Relational',
        'storage_objects': 'customers, products, reviews',
        'context_restoration': 'Two key-based joins in this example',
        'main_strength': 'Explicit structure and integrity controls',
    },
    {
        'design': 'Document',
        'storage_objects': 'One review document',
        'context_restoration': 'Nested fields retrieved with the record',
        'main_strength': 'Natural representation of semi-structured context',
    },
])

show_table(joined_review)
show_table(comparison)

## D.3 Building a synthetic e-commerce document database

The notebook creates three collections. `products` stores category-specific attributes, `reviews` stores nested customer context and optional review metadata, and `transactions` stores one order per document with a bounded array of line items. The records are synthetic and are designed for instruction rather than business benchmarking.

In [ ]:
# ============================================================
# D.3.1 Generate products, reviews, and transaction documents
# ============================================================
START_DATE = datetime(2026, 1, 1)
END_DATE = datetime(2026, 6, 30, 23, 59, 59)
DATE_SPAN_DAYS = (END_DATE.date() - START_DATE.date()).days + 1

product_specs = [
    {
        'name': 'Wireless Earbuds', 'category': 'Electronics', 'brand': 'Auralink', 'price': 79.00,
        'attributes': {'battery_hours': 8, 'connectivity': 'Bluetooth 5.3', 'water_resistant': True},
        'tags': ['audio', 'portable'], 'quality_mean': 4.3, 'demand_weight': 1.5,
        'complaints': ['sound', 'fit', 'battery'],
    },
    {
        'name': 'Smart Speaker', 'category': 'Electronics', 'brand': 'HomeWave', 'price': 99.00,
        'attributes': {'voice_assistant': True, 'connectivity': 'Wi-Fi', 'color_options': 3},
        'tags': ['smart_home', 'audio'], 'quality_mean': 2.6, 'demand_weight': 2.7,
        'complaints': ['connectivity', 'setup', 'privacy'],
    },
    {
        'name': 'Portable Charger', 'category': 'Electronics', 'brand': 'VoltGo', 'price': 39.00,
        'attributes': {'capacity_mah': 20000, 'ports': 3, 'fast_charge': True},
        'tags': ['travel', 'power'], 'quality_mean': 2.9, 'demand_weight': 2.4,
        'complaints': ['battery', 'durability', 'charging_speed'],
    },
    {
        'name': 'Webcam', 'category': 'Electronics', 'brand': 'ClearMeet', 'price': 69.00,
        'attributes': {'resolution': '1080p', 'microphone': True, 'privacy_shutter': True},
        'tags': ['office', 'video'], 'quality_mean': 4.0, 'demand_weight': 1.2,
        'complaints': ['image_quality', 'microphone', 'setup'],
    },
    {
        'name': 'Running Shoes', 'category': 'Footwear', 'brand': 'StrideLab', 'price': 110.00,
        'attributes': {'material': 'mesh', 'sizes': [6, 7, 8, 9, 10, 11, 12], 'wide_fit': True},
        'tags': ['running', 'fitness'], 'quality_mean': 4.4, 'demand_weight': 1.8,
        'complaints': ['fit', 'comfort', 'durability'],
    },
    {
        'name': 'Trail Shoes', 'category': 'Footwear', 'brand': 'PeakStep', 'price': 130.00,
        'attributes': {'material': 'synthetic', 'sizes': [7, 8, 9, 10, 11, 12], 'waterproof': True},
        'tags': ['trail', 'outdoor'], 'quality_mean': 3.7, 'demand_weight': 1.1,
        'complaints': ['traction', 'fit', 'weight'],
    },
    {
        'name': 'Casual Sneakers', 'category': 'Footwear', 'brand': 'MetroWalk', 'price': 75.00,
        'attributes': {'material': 'canvas', 'sizes': [6, 7, 8, 9, 10, 11], 'machine_washable': False},
        'tags': ['casual', 'lifestyle'], 'quality_mean': 3.9, 'demand_weight': 1.6,
        'complaints': ['fit', 'comfort', 'color'],
    },
    {
        'name': 'Hiking Boots', 'category': 'Footwear', 'brand': 'PeakStep', 'price': 155.00,
        'attributes': {'material': 'leather', 'sizes': [7, 8, 9, 10, 11, 12, 13], 'waterproof': True},
        'tags': ['hiking', 'outdoor'], 'quality_mean': 3.2, 'demand_weight': 1.0,
        'complaints': ['weight', 'break_in', 'waterproofing'],
    },
    {
        'name': 'Office Chair', 'category': 'Home', 'brand': 'ErgoNest', 'price': 249.00,
        'attributes': {'material': 'mesh', 'assembly_required': True, 'adjustable_arms': True},
        'tags': ['office', 'furniture'], 'quality_mean': 2.7, 'demand_weight': 2.1,
        'complaints': ['assembly', 'comfort', 'durability'],
    },
    {
        'name': 'Desk Lamp', 'category': 'Home', 'brand': 'LumaDesk', 'price': 45.00,
        'attributes': {'brightness_levels': 5, 'usb_port': True, 'bulb_type': 'LED'},
        'tags': ['office', 'lighting'], 'quality_mean': 4.2, 'demand_weight': 1.3,
        'complaints': ['brightness', 'controls', 'durability'],
    },
    {
        'name': 'Air Purifier', 'category': 'Home', 'brand': 'PureRoom', 'price': 189.00,
        'attributes': {'room_sq_ft': 450, 'filter_type': 'HEPA', 'smart_sensor': True},
        'tags': ['home', 'air_quality'], 'quality_mean': 3.5, 'demand_weight': 1.4,
        'complaints': ['noise', 'filter_cost', 'sensor'],
    },
    {
        'name': 'Coffee Maker', 'category': 'Home', 'brand': 'MorningCraft', 'price': 129.00,
        'attributes': {'capacity_cups': 12, 'programmable': True, 'carafe': 'thermal'},
        'tags': ['kitchen', 'coffee'], 'quality_mean': 3.8, 'demand_weight': 1.7,
        'complaints': ['temperature', 'leak', 'controls'],
    },
    {
        'name': 'Face Serum', 'category': 'Beauty', 'brand': 'LuminaSkin', 'price': 48.00,
        'attributes': {'volume_ml': 30, 'skin_type': ['normal', 'dry'], 'fragrance_free': True},
        'tags': ['skincare', 'hydration'], 'quality_mean': 4.1, 'demand_weight': 1.5,
        'complaints': ['irritation', 'texture', 'packaging'],
    },
    {
        'name': 'Moisturizer', 'category': 'Beauty', 'brand': 'LuminaSkin', 'price': 36.00,
        'attributes': {'volume_ml': 50, 'skin_type': ['normal', 'sensitive'], 'spf': 0},
        'tags': ['skincare', 'hydration'], 'quality_mean': 4.0, 'demand_weight': 1.7,
        'complaints': ['texture', 'irritation', 'scent'],
    },
    {
        'name': 'Shampoo', 'category': 'Beauty', 'brand': 'RootKind', 'price': 22.00,
        'attributes': {'volume_ml': 350, 'hair_type': ['normal', 'oily'], 'sulfate_free': True},
        'tags': ['haircare', 'daily_use'], 'quality_mean': 2.9, 'demand_weight': 2.0,
        'complaints': ['scent', 'dryness', 'packaging'],
    },
    {
        'name': 'Sunscreen', 'category': 'Beauty', 'brand': 'SunGuard', 'price': 28.00,
        'attributes': {'volume_ml': 75, 'spf': 50, 'water_resistant_minutes': 80},
        'tags': ['skincare', 'sun_protection'], 'quality_mean': 3.6, 'demand_weight': 1.8,
        'complaints': ['texture', 'white_cast', 'irritation'],
    },
]

product_documents = []
generation_profiles = {}
for position, spec in enumerate(product_specs, start=1):
    product_id = f'P{position:03d}'
    product_documents.append({
        '_id': product_id,
        'name': spec['name'],
        'category': spec['category'],
        'brand': spec['brand'],
        'price': spec['price'],
        'attributes': deepcopy(spec['attributes']),
        'tags': list(spec['tags']),
        'active': True,
        'created_at': datetime(2025, 11, 1) + timedelta(days=position),
    })
    generation_profiles[product_id] = {
        'quality_mean': spec['quality_mean'],
        'demand_weight': spec['demand_weight'],
        'complaints': list(spec['complaints']),
    }

customer_ids = [f'C{value:04d}' for value in range(1, 241)]
loyalty_levels = ['Standard', 'Silver', 'Gold']
regions = ['West', 'Midwest', 'South', 'Northeast']
customer_profiles = {
    customer_id: {
        'customer_id': customer_id,
        'loyalty_level': str(RNG.choice(loyalty_levels, p=[0.55, 0.28, 0.17])),
        'region': str(RNG.choice(regions, p=[0.32, 0.21, 0.27, 0.20])),
    }
    for customer_id in customer_ids
}

positive_templates = [
    'The {name} works well and feels reliable.',
    'Good value, easy to use, and consistent so far.',
    'The {name} met my expectations and arrived in good condition.',
    'I would consider buying this product again.',
]
neutral_templates = [
    'The {name} does the basic job, although the experience is average.',
    'The product is acceptable, but there are a few trade-offs.',
    'It works, but I expected a more polished experience.',
]
aspect_sentences = {
    'assembly': 'Assembly took much longer than expected.',
    'battery': 'The battery performance is below expectations.',
    'break_in': 'The break-in period is uncomfortable.',
    'brightness': 'The brightness is not sufficient for my workspace.',
    'charging_speed': 'Charging is slower than advertised.',
    'color': 'The color differs from the listing.',
    'comfort': 'The product becomes uncomfortable during longer use.',
    'connectivity': 'The connection drops too often.',
    'controls': 'The controls are confusing to operate.',
    'dryness': 'The product leaves my hair feeling dry.',
    'durability': 'The product showed wear sooner than expected.',
    'filter_cost': 'Replacement filters are more expensive than expected.',
    'fit': 'The fit is inconsistent with the size guide.',
    'image_quality': 'The image quality is weaker than expected.',
    'irritation': 'The product caused irritation for me.',
    'leak': 'The unit began to leak during normal use.',
    'microphone': 'The microphone does not sound clear.',
    'noise': 'The unit is louder than expected.',
    'packaging': 'The packaging is inconvenient and wasteful.',
    'privacy': 'The privacy controls are not clear enough.',
    'scent': 'The scent is stronger than expected.',
    'sensor': 'The sensor readings appear inconsistent.',
    'setup': 'Setup was more difficult than it should be.',
    'sound': 'The sound quality is inconsistent.',
    'temperature': 'The brewing temperature is not consistent.',
    'texture': 'The texture feels unpleasant on the skin.',
    'traction': 'Traction is weaker on wet surfaces.',
    'waterproofing': 'Water resistance did not meet expectations.',
    'weight': 'The product feels heavier than expected.',
    'white_cast': 'The formula leaves a noticeable white cast.',
}

review_documents = []
review_number = 1
for product in product_documents:
    product_id = product['_id']
    profile = generation_profiles[product_id]
    review_count = max(18, int(22 + 14 * profile['demand_weight'] + RNG.integers(-4, 5)))
    for _ in range(review_count):
        rating = int(np.clip(np.rint(RNG.normal(profile['quality_mean'], 0.9)), 1, 5))
        if rating <= 2:
            aspect_count = min(len(profile['complaints']), int(RNG.integers(1, 3)))
            aspects = list(RNG.choice(profile['complaints'], size=aspect_count, replace=False))
            review_text = ' '.join(aspect_sentences[aspect] for aspect in aspects)
        elif rating == 3:
            aspects = [str(RNG.choice(profile['complaints']))]
            review_text = str(RNG.choice(neutral_templates)).format(name=product['name'])
        else:
            aspects = list(RNG.choice(['value', 'ease_of_use', 'quality', 'shipping'], size=2, replace=False))
            review_text = str(RNG.choice(positive_templates)).format(name=product['name'])

        customer_id = str(RNG.choice(customer_ids))
        source = str(RNG.choice(['web', 'mobile_app', 'store_kiosk'], p=[0.48, 0.42, 0.10]))
        created_at = START_DATE + timedelta(
            days=int(RNG.integers(0, DATE_SPAN_DAYS)),
            hours=int(RNG.integers(0, 24)),
            minutes=int(RNG.integers(0, 60)),
        )
        document = {
            '_id': f'R{review_number:05d}',
            'product_id': product_id,
            'customer': deepcopy(customer_profiles[customer_id]),
            'rating': rating,
            'review_text': review_text,
            'aspects': aspects,
            'source': source,
            'language': str(RNG.choice(['en', 'es', 'ko'], p=[0.86, 0.08, 0.06])),
            'verified_purchase': bool(RNG.random() < 0.82),
            'helpful_votes': int(RNG.poisson(1.4)),
            'created_at': created_at,
        }
        if source != 'store_kiosk':
            document['device'] = {
                'type': str(RNG.choice(['phone', 'tablet', 'desktop'], p=[0.58, 0.10, 0.32])),
                'app_version': f'{int(RNG.integers(4, 7))}.{int(RNG.integers(0, 10))}',
            }
        if RNG.random() < 0.36:
            document['survey'] = {
                'would_recommend': bool(rating >= 4),
                'nps': int(np.clip((rating - 1) * 2 + RNG.integers(-1, 3), 0, 10)),
            }
        if RNG.random() < 0.18:
            document['campaign_id'] = str(RNG.choice(['SPRING26', 'LOYALTY26', 'APP26']))
        review_documents.append(document)
        review_number += 1

product_ids = [document['_id'] for document in product_documents]
product_lookup = {document['_id']: document for document in product_documents}
demand_probabilities = np.array([
    generation_profiles[product_id]['demand_weight'] for product_id in product_ids
], dtype=float)
demand_probabilities = demand_probabilities / demand_probabilities.sum()

transaction_documents = []
for transaction_number in range(1, 901):
    customer_id = str(RNG.choice(customer_ids))
    purchased_at = START_DATE + timedelta(
        days=int(RNG.integers(0, DATE_SPAN_DAYS)),
        hours=int(RNG.integers(7, 23)),
        minutes=int(RNG.integers(0, 60)),
    )
    item_count = int(RNG.choice([1, 2, 3, 4], p=[0.42, 0.34, 0.18, 0.06]))
    chosen_products = list(RNG.choice(product_ids, size=item_count, replace=False, p=demand_probabilities))
    items = []
    for product_id in chosen_products:
        quantity = int(RNG.choice([1, 2, 3], p=[0.78, 0.18, 0.04]))
        unit_price = float(product_lookup[product_id]['price'])
        discount_pct = float(RNG.choice([0.00, 0.10, 0.15, 0.20], p=[0.68, 0.17, 0.10, 0.05]))
        line_revenue = round(quantity * unit_price * (1 - discount_pct), 2)
        items.append({
            'product_id': product_id,
            'quantity': quantity,
            'unit_price': unit_price,
            'discount_pct': discount_pct,
            'line_revenue': line_revenue,
        })
    transaction = {
        '_id': f'T{transaction_number:05d}',
        'customer_id': customer_id,
        'purchased_at': purchased_at,
        'channel': str(RNG.choice(['web', 'mobile_app', 'store'], p=[0.44, 0.38, 0.18])),
        'items': items,
        'total_amount': round(sum(item['line_revenue'] for item in items), 2),
        'payment_method': str(RNG.choice(['card', 'digital_wallet', 'gift_card'], p=[0.68, 0.27, 0.05])),
        'shipping': {
            'region': customer_profiles[customer_id]['region'],
            'expedited': bool(RNG.random() < 0.14),
        },
    }
    if RNG.random() < 0.22:
        transaction['campaign_id'] = str(RNG.choice(['SPRING26', 'LOYALTY26', 'APP26']))
    transaction_documents.append(transaction)

save_jsonl(product_documents, DATA_DIR / 'products_source.jsonl')
save_jsonl(review_documents, DATA_DIR / 'reviews_source.jsonl')
save_jsonl(transaction_documents, DATA_DIR / 'transactions_source.jsonl')

print(f'Products: {len(product_documents):,}')
print(f'Reviews: {len(review_documents):,}')
print(f'Transactions: {len(transaction_documents):,}')
print(f'Embedded transaction line items: {sum(len(doc["items"]) for doc in transaction_documents):,}')

In [ ]:
# ============================================================
# D.3.2 Inspect schema flexibility before loading the database
# ============================================================
product_attribute_map = pd.DataFrame([
    {
        'product_id': document['_id'],
        'name': document['name'],
        'category': document['category'],
        'attribute_keys': ', '.join(sorted(document['attributes'].keys())),
        'attribute_count': len(document['attributes']),
    }
    for document in product_documents
])

optional_review_fields = pd.DataFrame([
    {
        'field': field,
        'documents_with_field': sum(field in document for document in review_documents),
        'coverage': np.mean([field in document for document in review_documents]),
    }
    for field in ['device', 'survey', 'campaign_id']
])

show_table(product_attribute_map, rows=20)
show_table(optional_review_fields, rows=10)
print('\nExample product with category-specific attributes:')
print_document(product_documents[0])
print('\nExample transaction with an embedded line-item array:')
print_document(transaction_documents[0])

In [ ]:
# ============================================================
# D.3.3 Create the in-memory database, collections, and documents
# ============================================================
client = mongomock.MongoClient()
db = client['ecommerce']
products = db['products']
reviews = db['reviews']
transactions = db['transactions']

products.insert_many(deepcopy(product_documents))
reviews.insert_many(deepcopy(review_documents))
transactions.insert_many(deepcopy(transaction_documents))

collection_summary = pd.DataFrame([
    {
        'database': db.name,
        'collection': collection_name,
        'document_count': db[collection_name].count_documents({}),
        'example_grain': {
            'products': 'one product',
            'reviews': 'one customer review',
            'transactions': 'one purchase transaction with line items',
        }[collection_name],
    }
    for collection_name in sorted(db.list_collection_names())
])

show_table(collection_summary, rows=10)
print('Collection names:', sorted(db.list_collection_names()))
print('\nOne stored review document:')
print_document(reviews.find_one())

## D.4 CRUD operations and nested queries

CRUD stands for create, read, update, and delete. The same operations support data ingestion, correction, enrichment, and analytical retrieval. In production systems, destructive operations should be protected by permissions, backups, and review procedures. The examples below use a clearly labeled temporary document so that the synthetic source data remain unchanged.

In [ ]:
# ============================================================
# D.4.1 Create and read a temporary review
# ============================================================
demo_review = {
    '_id': 'R_CLASSROOM_DEMO',
    'product_id': 'P001',
    'customer': {
        'customer_id': 'C_DEMO',
        'loyalty_level': 'Standard',
        'region': 'West',
    },
    'rating': 3,
    'review_text': 'Useful classroom record for demonstrating CRUD operations.',
    'aspects': ['documentation'],
    'source': 'web',
    'language': 'en',
    'verified_purchase': False,
    'helpful_votes': 0,
    'created_at': datetime(2026, 6, 30, 12, 0),
}

insert_result = reviews.insert_one(deepcopy(demo_review))
print('Inserted ID:', insert_result.inserted_id)
print_document(reviews.find_one({'_id': 'R_CLASSROOM_DEMO'}))

low_rating_query = {
    'rating': {'$lte': 2},
    'source': {'$in': ['web', 'mobile_app']},
}
low_rating_projection = {
    '_id': 1,
    'product_id': 1,
    'rating': 1,
    'review_text': 1,
    'source': 1,
    'created_at': 1,
}

low_rating_sample = cursor_df(
    reviews.find(low_rating_query, low_rating_projection)
    .sort([('rating', ASCENDING), ('created_at', DESCENDING)])
    .limit(8)
)
show_table(low_rating_sample, rows=10)

In [ ]:
# ============================================================
# D.4.2 Query nested fields and arrays
# ============================================================
nested_query = {
    'customer.loyalty_level': 'Gold',
    'aspects': {'$in': ['connectivity', 'battery']},
    'rating': {'$lte': 3},
}
nested_projection = {
    '_id': 1,
    'product_id': 1,
    'customer.loyalty_level': 1,
    'customer.region': 1,
    'rating': 1,
    'aspects': 1,
    'source': 1,
}

nested_results = cursor_df(
    reviews.find(nested_query, nested_projection)
    .sort('created_at', DESCENDING)
    .limit(10)
)

array_examples = pd.DataFrame([
    {'query_goal': 'Any listed complaint aspect', 'filter_fragment': "{'aspects': {'$in': ['battery', 'connectivity']}}"},
    {'query_goal': 'A specific nested loyalty level', 'filter_fragment': "{'customer.loyalty_level': 'Gold'}"},
    {'query_goal': 'Several conditions together', 'filter_fragment': "{'rating': {'$lte': 3}, 'source': 'mobile_app'}"},
])

show_table(nested_results, rows=10)
show_table(array_examples, rows=10)

In [ ]:
# ============================================================
# D.4.3 Update, verify, and delete the temporary document
# ============================================================
update_result = reviews.update_one(
    {'_id': 'R_CLASSROOM_DEMO'},
    {
        '$set': {
            'analysis.status': 'reviewed',
            'analysis.priority': 'low',
        },
        '$inc': {'helpful_votes': 1},
        '$addToSet': {'aspects': 'training_example'},
    },
)

updated_demo = reviews.find_one({'_id': 'R_CLASSROOM_DEMO'})
print('Matched documents:', update_result.matched_count)
print('Modified documents:', update_result.modified_count)
print_document(updated_demo)

before_delete = reviews.count_documents({'_id': 'R_CLASSROOM_DEMO'})
delete_result = reviews.delete_one({'_id': 'R_CLASSROOM_DEMO'})
after_delete = reviews.count_documents({'_id': 'R_CLASSROOM_DEMO'})

crud_audit = pd.DataFrame([{
    'count_before_delete': before_delete,
    'deleted_count': delete_result.deleted_count,
    'count_after_delete': after_delete,
    'source_review_count_restored': reviews.count_documents({}) == len(review_documents),
}])
show_table(crud_audit)

## D.5 Designing documents: embedding versus referencing

Embedding keeps related information inside one parent document. Referencing stores related information separately and connects it through an identifier. Embedding is natural when the child data are small, bounded, and usually read with the parent. Referencing is safer when the related records grow without a clear bound, are updated independently, are shared, or need their own analytical queries.

In [ ]:
# ============================================================
# D.5.1 Compare embedding and referencing decisions
# ============================================================
design_guide = pd.DataFrame([
    {
        'design_question': 'Are the records usually read together?',
        'favor_embedding': 'Yes, the child context is normally needed with the parent',
        'favor_referencing': 'The child is often queried by itself',
    },
    {
        'design_question': 'Can the related data grow without a bound?',
        'favor_embedding': 'No, the amount is small and bounded',
        'favor_referencing': 'Yes, the history can continue growing',
    },
    {
        'design_question': 'How often is the related data updated?',
        'favor_embedding': 'Rarely or together with the parent',
        'favor_referencing': 'Frequently and independently',
    },
    {
        'design_question': 'Is the related data reused?',
        'favor_embedding': 'It belongs to one parent only',
        'favor_referencing': 'It is shared across many records',
    },
    {
        'design_question': 'What does analytics require?',
        'favor_embedding': 'Retrieve one complete bounded record',
        'favor_referencing': 'Filter, group, join, or model the child records independently',
    },
])

embedded_transaction = transactions.find_one({}, {'_id': 1, 'customer_id': 1, 'items': 1, 'total_amount': 1})
referenced_review = reviews.find_one({}, {'_id': 1, 'product_id': 1, 'rating': 1, 'review_text': 1})
referenced_product = products.find_one({'_id': referenced_review['product_id']}, {'_id': 1, 'name': 1, 'category': 1})

show_table(design_guide, rows=10)
print('\nEmbedded line items inside one transaction:')
print_document(embedded_transaction)
print('\nReferenced review and product documents:')
print_document({'review': referenced_review, 'product': referenced_product})

In [ ]:
# ============================================================
# D.5.2 Observe how an embedded history increases document size
# ============================================================
base_product = products.find_one({'_id': 'P001'})
review_stub = {
    'review_id': 'R_STUB',
    'rating': 4,
    'review_text': 'A short review retained inside a product preview.',
    'created_at': datetime(2026, 1, 1),
}

size_rows = []
for embedded_count in [0, 5, 20, 100, 500, 1000]:
    candidate = deepcopy(base_product)
    candidate['recent_reviews'] = [
        {**review_stub, 'review_id': f'R_STUB_{index:04d}'}
        for index in range(embedded_count)
    ]
    size_bytes = bson_size_bytes(candidate)
    size_rows.append({
        'embedded_review_count': embedded_count,
        'bson_size_bytes': size_bytes,
        'bson_size_kib': size_bytes / 1024,
        'share_of_16_mib_limit': size_bytes / (16 * 1024 * 1024),
    })

size_growth = pd.DataFrame(size_rows)
show_table(size_growth, rows=10)

plt.figure(figsize=(7.5, 4.5))
plt.plot(size_growth['embedded_review_count'], size_growth['bson_size_kib'], marker='o')
plt.xlabel('Number of embedded review previews')
plt.ylabel('Encoded document size (KiB)')
plt.title('Unbounded embedded histories make a parent document grow')
document_size_chart_path = save_current_figure('embedded_document_size_growth.png')

## D.6 Indexes and access patterns

An index should support a repeated filter or sort that matters to the workflow. A compound index should usually begin with fields used to narrow the search, followed by fields used for sorting or additional filtering. Too many indexes increase storage and write costs.

The in-memory mock records index definitions but does not reproduce MongoDB's query planner or production timing. Therefore, the notebook documents index intent rather than presenting mock timing as a database benchmark.

In [ ]:
# ============================================================
# D.6.1 Create indexes that correspond to repeated queries
# ============================================================
reviews.create_index(
    [('product_id', ASCENDING), ('created_at', DESCENDING), ('rating', ASCENDING)],
    name='product_recent_rating',
)
reviews.create_index(
    [('source', ASCENDING), ('created_at', DESCENDING)],
    name='source_recent',
)
transactions.create_index(
    [('purchased_at', DESCENDING)],
    name='purchase_time',
)
transactions.create_index(
    [('items.product_id', ASCENDING), ('purchased_at', DESCENDING)],
    name='line_item_product_recent',
)

index_rows = []
for collection_name, collection in [('reviews', reviews), ('transactions', transactions)]:
    for index_name, information in collection.index_information().items():
        index_rows.append({
            'collection': collection_name,
            'index_name': index_name,
            'key_order': information.get('key'),
            'unique': information.get('unique', index_name == '_id_'),
        })
index_catalog = pd.DataFrame(index_rows)
show_table(index_catalog, rows=20)

In [ ]:
# ============================================================
# D.6.2 Translate access patterns into index intent
# ============================================================
access_pattern_guide = pd.DataFrame([
    {
        'access_pattern': 'Recent low-rating reviews for one product',
        'filter_and_sort': 'product_id equality, created_at descending, rating condition',
        'candidate_index': [('product_id', 1), ('created_at', -1), ('rating', 1)],
    },
    {
        'access_pattern': 'Recent reviews from one channel',
        'filter_and_sort': 'source equality, created_at descending',
        'candidate_index': [('source', 1), ('created_at', -1)],
    },
    {
        'access_pattern': 'Transactions containing one product before a cutoff',
        'filter_and_sort': 'items.product_id equality, purchased_at range',
        'candidate_index': [('items.product_id', 1), ('purchased_at', -1)],
    },
])

indexed_query = {
    'product_id': 'P002',
    'rating': {'$lte': 2},
    'created_at': {'$lte': datetime(2026, 5, 31, 23, 59, 59)},
}
indexed_query_result = cursor_df(
    reviews.find(indexed_query, {'_id': 1, 'product_id': 1, 'rating': 1, 'created_at': 1})
    .sort('created_at', DESCENDING)
    .limit(8)
)

show_table(access_pattern_guide, rows=10)
show_table(indexed_query_result, rows=10)
print('Production note: inspect execution statistics on a real MongoDB deployment before drawing performance conclusions.')

## D.7 Aggregation pipelines

An aggregation pipeline passes documents through ordered stages. `$match` filters, `$project` reshapes, `$group` summarizes, `$unwind` expands arrays, `$sort` orders results, and `$lookup` adds context from another collection. Filtering early is often a useful design principle because it reduces the records passed to later stages.

In [ ]:
# ============================================================
# D.7.1 Count low-rating reviews by product
# ============================================================
low_rating_pipeline = [
    {'$match': {'rating': {'$lte': 2}}},
    {'$group': {
        '_id': '$product_id',
        'low_review_count': {'$sum': 1},
        'average_low_rating': {'$avg': '$rating'},
    }},
    {'$sort': {'low_review_count': -1, '_id': 1}},
    {'$limit': 10},
]

low_rating_by_product = cursor_df(reviews.aggregate(low_rating_pipeline))
if not low_rating_by_product.empty:
    low_rating_by_product = low_rating_by_product.rename(columns={'_id': 'product_id'})
show_table(low_rating_by_product, rows=15)

pipeline_flow = pd.DataFrame([
    {'stage': '$match', 'documents_or_groups': reviews.count_documents({'rating': {'$lte': 2}}), 'meaning': 'Low-rating review documents retained'},
    {'stage': '$group', 'documents_or_groups': len(reviews.distinct('product_id', {'rating': {'$lte': 2}})), 'meaning': 'Product-level groups created'},
    {'stage': '$sort and $limit', 'documents_or_groups': len(low_rating_by_product), 'meaning': 'Highest-count groups returned'},
])
show_table(pipeline_flow, rows=10)

In [ ]:
# ============================================================
# D.7.2 Expand the aspects array and summarize complaint themes
# ============================================================
complaint_aspect_pipeline = [
    {'$match': {'rating': {'$lte': 2}}},
    {'$unwind': '$aspects'},
    {'$group': {
        '_id': '$aspects',
        'mention_count': {'$sum': 1},
        'product_ids': {'$addToSet': '$product_id'},
    }},
    {'$sort': {'mention_count': -1, '_id': 1}},
]

complaint_aspects_raw = list(reviews.aggregate(complaint_aspect_pipeline))
complaint_aspects = pd.DataFrame([
    {
        'aspect': row['_id'],
        'mention_count': row['mention_count'],
        'product_count': len(row['product_ids']),
    }
    for row in complaint_aspects_raw
])
show_table(complaint_aspects, rows=15)

In [ ]:
# ============================================================
# D.7.3 Add product context with $lookup
# ============================================================
lookup_pipeline = [
    {'$match': {'rating': {'$lte': 2}}},
    {'$group': {
        '_id': '$product_id',
        'low_review_count': {'$sum': 1},
        'average_low_rating': {'$avg': '$rating'},
    }},
    {'$lookup': {
        'from': 'products',
        'localField': '_id',
        'foreignField': '_id',
        'as': 'product',
    }},
    {'$unwind': '$product'},
    {'$project': {
        '_id': 0,
        'product_id': '$_id',
        'product_name': '$product.name',
        'category': '$product.category',
        'low_review_count': 1,
        'average_low_rating': 1,
    }},
    {'$sort': {'low_review_count': -1, 'product_id': 1}},
    {'$limit': 10},
]

low_rating_with_context = cursor_df(reviews.aggregate(lookup_pipeline))
show_table(low_rating_with_context, rows=15)

## D.8 Moving focused document data into pandas

The database should reduce the data before pandas loads it into memory. Filters define the eligible records, projections retain only needed fields, and aggregation can establish the intended analytical grain. Python then supports visualization, text analysis, modeling, and communication.

In [ ]:
# ============================================================
# D.8.1 Filter and project before conversion to a DataFrame
# ============================================================
AS_OF_DATE = datetime(2026, 5, 31, 23, 59, 59)

focused_cursor = reviews.find(
    {
        'rating': {'$lte': 3},
        'created_at': {'$lte': AS_OF_DATE},
    },
    {
        '_id': 0,
        'product_id': 1,
        'customer.loyalty_level': 1,
        'customer.region': 1,
        'rating': 1,
        'review_text': 1,
        'aspects': 1,
        'source': 1,
        'language': 1,
        'created_at': 1,
    },
)
focused_reviews_df = cursor_df(focused_cursor)
focused_reviews_df['created_at'] = pd.to_datetime(focused_reviews_df['created_at'], utc=True)

all_reviews_df = cursor_df(reviews.find())
full_memory_kib = all_reviews_df.memory_usage(deep=True).sum() / 1024
focused_memory_kib = focused_reviews_df.memory_usage(deep=True).sum() / 1024

transfer_audit = pd.DataFrame([{
    'database_review_documents': reviews.count_documents({}),
    'focused_documents_moved_to_pandas': len(focused_reviews_df),
    'full_dataframe_memory_kib': full_memory_kib,
    'focused_dataframe_memory_kib': focused_memory_kib,
    'as_of_date': AS_OF_DATE.date().isoformat(),
}])

show_table(focused_reviews_df, rows=10)
show_table(transfer_audit)

In [ ]:
# ============================================================
# D.8.2 Summarize and visualize the focused result
# ============================================================
source_summary = (
    focused_reviews_df
    .groupby('source', as_index=False)
    .agg(
        review_count=('rating', 'size'),
        average_rating=('rating', 'mean'),
        low_rating_share=('rating', lambda values: np.mean(values <= 2)),
    )
    .sort_values('review_count', ascending=False)
)

focused_reviews_path = OUT_DIR / 'focused_reviews_for_python.csv'
source_summary_path = OUT_DIR / 'review_source_summary.csv'
focused_reviews_df.to_csv(focused_reviews_path, index=False)
source_summary.to_csv(source_summary_path, index=False)

show_table(source_summary, rows=10)

plt.figure(figsize=(7.5, 4.5))
plt.bar(source_summary['source'], source_summary['low_rating_share'])
plt.xlabel('Review source')
plt.ylabel('Share of ratings at or below 2')
plt.title('Low-rating share in the focused review extract')
source_chart_path = save_current_figure('low_rating_share_by_source.png')

## D.9 Data quality, schema validation, and integrity checks

Flexible schema does not mean absent schema. Analytics still requires consistent names, types, allowed values, and definitions. The following examples deliberately introduce inconsistent raw records. The notebook profiles the fields, validates a canonical contract, and separately audits references because a document database does not automatically enforce every business relationship. The executable example uses standard JSON Schema after converting BSON values into JSON-safe values. MongoDB collection validators use the related `$jsonSchema` form with BSON-aware types.

In [ ]:
# ============================================================
# D.9.1 Create intentionally inconsistent raw review examples
# ============================================================
raw_quality_examples = [
    {
        '_id': 'Q001',
        'product_id': 'P001',
        'customer': {'customer_id': 'C0001', 'loyalty_level': 'Gold', 'region': 'West'},
        'rating': 5,
        'review_text': 'A valid example.',
        'source': 'web',
        'language': 'en',
        'created_at': datetime(2026, 3, 2, 9, 0),
    },
    {
        '_id': 'Q002',
        'product_id': 'P002',
        'customer': {'customer_id': 'C0002', 'loyalty_level': 'Silver', 'region': 'South'},
        'rating': '2',
        'review_text': 'Rating was stored as text.',
        'source': 'mobile_app',
        'language': 'en',
        'created_at': '2026-03-03T10:00:00Z',
    },
    {
        '_id': 'Q003',
        'product_id': 'P003',
        'customer': {'customer_id': 'C0003'},
        'stars': 1,
        'review_text': 'The expected rating field is missing.',
        'source': 'web',
        'language': 'en',
        'created_at': datetime(2026, 3, 4, 11, 0),
    },
    {
        '_id': 'Q004',
        'product_id': 'P999',
        'customer': {'customer_id': 'C0004', 'loyalty_level': 'Standard', 'region': 'Northeast'},
        'rating': 2,
        'review_text': 'The referenced product does not exist.',
        'source': 'store_kiosk',
        'language': 'en',
        'created_at': datetime(2026, 3, 5, 12, 0),
    },
    {
        '_id': 'Q005',
        'product_id': 'P004',
        'customer': {'customer_id': 'C0005', 'loyalty_level': 'Standard', 'region': 'Midwest'},
        'rating': 6,
        'review_txt': 'The text field is misspelled and the rating is outside the allowed range.',
        'source': 'unknown_channel',
        'language': 'xx',
        'created_at': datetime(2026, 3, 6, 13, 0),
    },
]

raw_reviews = db['raw_reviews_quality_lab']
raw_reviews.insert_many(deepcopy(raw_quality_examples))
raw_profile = field_profile(raw_reviews.find())

show_table(raw_profile, rows=40)

In [ ]:
# ============================================================
# D.9.2 Validate a canonical review contract
# ============================================================
review_schema = {
    '$schema': 'https://json-schema.org/draft/2020-12/schema',
    'type': 'object',
    'required': [
        '_id', 'product_id', 'customer', 'rating', 'review_text',
        'source', 'language', 'created_at'
    ],
    'properties': {
        '_id': {'type': 'string', 'minLength': 1},
        'product_id': {'type': 'string', 'pattern': '^P[0-9]{3}$'},
        'customer': {
            'type': 'object',
            'required': ['customer_id', 'loyalty_level', 'region'],
            'properties': {
                'customer_id': {'type': 'string'},
                'loyalty_level': {'enum': ['Standard', 'Silver', 'Gold']},
                'region': {'enum': ['West', 'Midwest', 'South', 'Northeast']},
            },
            'additionalProperties': True,
        },
        'rating': {'type': 'integer', 'minimum': 1, 'maximum': 5},
        'review_text': {'type': 'string', 'minLength': 1},
        'source': {'enum': ['web', 'mobile_app', 'store_kiosk']},
        'language': {'enum': ['en', 'es', 'ko']},
        'created_at': {'type': 'string', 'format': 'date-time'},
    },
    'additionalProperties': True,
}

validator = Draft202012Validator(review_schema, format_checker=FormatChecker())
validation_documents = [reviews.find_one()] + list(raw_reviews.find())
validation_rows = []
for document in validation_documents:
    safe_document = json_safe(document)
    errors = sorted(validator.iter_errors(safe_document), key=lambda error: list(error.path))
    validation_rows.append({
        'document_id': safe_document.get('_id'),
        'valid_schema': len(errors) == 0,
        'error_count': len(errors),
        'first_error': errors[0].message if errors else '',
    })

validation_report = pd.DataFrame(validation_rows)
schema_path = OUT_DIR / 'canonical_review_schema.json'
validation_report_path = OUT_DIR / 'review_schema_validation_report.csv'
save_json(review_schema, schema_path)
validation_report.to_csv(validation_report_path, index=False)

show_table(validation_report, rows=20)

In [ ]:
# ============================================================
# D.9.3 Audit referenced identifiers and document-size risk
# ============================================================
known_product_ids = set(products.distinct('_id'))
main_review_product_ids = set(reviews.distinct('product_id'))
raw_review_product_ids = set(raw_reviews.distinct('product_id'))
transaction_product_ids = {
    item['product_id']
    for transaction in transactions.find({}, {'items.product_id': 1})
    for item in transaction.get('items', [])
}

integrity_audit = pd.DataFrame([
    {
        'collection': 'reviews',
        'referenced_product_count': len(main_review_product_ids),
        'orphan_product_ids': sorted(main_review_product_ids - known_product_ids),
        'passes_reference_check': not (main_review_product_ids - known_product_ids),
    },
    {
        'collection': 'transactions.items',
        'referenced_product_count': len(transaction_product_ids),
        'orphan_product_ids': sorted(transaction_product_ids - known_product_ids),
        'passes_reference_check': not (transaction_product_ids - known_product_ids),
    },
    {
        'collection': 'raw_reviews_quality_lab',
        'referenced_product_count': len(raw_review_product_ids),
        'orphan_product_ids': sorted(raw_review_product_ids - known_product_ids),
        'passes_reference_check': not (raw_review_product_ids - known_product_ids),
    },
])

largest_transaction = max(transactions.find(), key=bson_size_bytes)
size_audit = pd.DataFrame([{
    'largest_transaction_id': largest_transaction['_id'],
    'line_item_count': len(largest_transaction['items']),
    'bson_size_bytes': bson_size_bytes(largest_transaction),
    'share_of_16_mib_limit': bson_size_bytes(largest_transaction) / (16 * 1024 * 1024),
}])

show_table(integrity_audit, rows=10)
show_table(size_audit)

## D.10 Optional MongoDB Atlas connection

The core notebook does not require a server. The next cell runs safely even when no connection string is available. To test a real Atlas deployment, place the URI in an environment variable named `MONGODB_URI`. Do not paste a password into a shared notebook, screenshot, output cell, or repository. Use a database account with only the privileges required for the exercise.

In [ ]:
# ============================================================
# D.10.1 Optional protected Atlas connection test
# ============================================================
atlas_uri = os.getenv('MONGODB_URI')

if not atlas_uri:
    atlas_status = {
        'connected': False,
        'message': 'Skipped: MONGODB_URI is not set. The in-memory exercises remain complete.',
    }
else:
    try:
        with PyMongoClient(atlas_uri, serverSelectionTimeoutMS=5000) as atlas_client:
            atlas_client.admin.command('ping')
        atlas_status = {
            'connected': True,
            'message': 'Ping succeeded. Credentials were read from the environment and were not printed.',
        }
    except Exception as error:
        atlas_status = {
            'connected': False,
            'message': f'Connection attempt failed: {type(error).__name__}. Review network access and credentials.',
        }

print(json.dumps(atlas_status, indent=2))

## D.11 Applied mini-case: customer feedback and purchase analytics

A retailer wants to identify products that sell strongly but receive a meaningful share of low ratings. The analysis uses only records available by an explicit as-of date. Review records are summarized at the product grain, transaction arrays are expanded and summarized at the same grain, and product metadata restores managerial context. The result is a prioritization aid for investigation, not an automatic product-removal rule.

In [ ]:
# ============================================================
# D.11.1 Build product-level review metrics as of the cutoff
# ============================================================
AS_OF_DATE = datetime(2026, 5, 31, 23, 59, 59)

review_summary_pipeline = [
    {'$match': {'created_at': {'$lte': AS_OF_DATE}}},
    {'$group': {
        '_id': '$product_id',
        'review_count': {'$sum': 1},
        'average_rating': {'$avg': '$rating'},
    }},
    {'$sort': {'_id': 1}},
]
low_review_summary_pipeline = [
    {'$match': {
        'created_at': {'$lte': AS_OF_DATE},
        'rating': {'$lte': 2},
    }},
    {'$group': {
        '_id': '$product_id',
        'low_review_count': {'$sum': 1},
    }},
    {'$sort': {'_id': 1}},
]

review_metrics = cursor_df(reviews.aggregate(review_summary_pipeline)).rename(columns={'_id': 'product_id'})
low_review_metrics = cursor_df(reviews.aggregate(low_review_summary_pipeline)).rename(columns={'_id': 'product_id'})
review_metrics = review_metrics.merge(low_review_metrics, on='product_id', how='left')
review_metrics['low_review_count'] = review_metrics['low_review_count'].fillna(0).astype(int)
review_metrics['low_rating_share'] = review_metrics['low_review_count'] / review_metrics['review_count']

future_review_count = reviews.count_documents({'created_at': {'$gt': AS_OF_DATE}})
show_table(review_metrics.sort_values('low_rating_share', ascending=False), rows=20)
print('Reviews excluded because they occur after the as-of date:', future_review_count)

In [ ]:
# ============================================================
# D.11.2 Expand line items and build product-level purchase metrics
# ============================================================
transaction_summary_pipeline = [
    {'$match': {'purchased_at': {'$lte': AS_OF_DATE}}},
    {'$unwind': '$items'},
    {'$group': {
        '_id': '$items.product_id',
        'units_sold': {'$sum': '$items.quantity'},
        'net_revenue': {'$sum': '$items.line_revenue'},
        'order_ids': {'$addToSet': '$_id'},
        'customer_ids': {'$addToSet': '$customer_id'},
    }},
    {'$sort': {'_id': 1}},
]

transaction_metric_rows = []
for row in transactions.aggregate(transaction_summary_pipeline):
    transaction_metric_rows.append({
        'product_id': row['_id'],
        'units_sold': row['units_sold'],
        'net_revenue': row['net_revenue'],
        'order_count': len(row['order_ids']),
        'unique_customers': len(row['customer_ids']),
    })
transaction_metrics = pd.DataFrame(transaction_metric_rows)

future_transaction_count = transactions.count_documents({'purchased_at': {'$gt': AS_OF_DATE}})
show_table(transaction_metrics.sort_values('net_revenue', ascending=False), rows=20)
print('Transactions excluded because they occur after the as-of date:', future_transaction_count)

In [ ]:
# ============================================================
# D.11.3 Combine evidence and rank products for investigation
# ============================================================
product_context = cursor_df(
    products.find({}, {'_id': 1, 'name': 1, 'category': 1, 'brand': 1, 'price': 1})
).rename(columns={'_id': 'product_id'})

product_scorecard = (
    product_context
    .merge(review_metrics, on='product_id', how='left', validate='one_to_one')
    .merge(transaction_metrics, on='product_id', how='left', validate='one_to_one')
)

numeric_fill_columns = [
    'review_count', 'average_rating', 'low_review_count', 'low_rating_share',
    'units_sold', 'net_revenue', 'order_count', 'unique_customers'
]
product_scorecard[numeric_fill_columns] = product_scorecard[numeric_fill_columns].fillna(0)

sales_cutoff = product_scorecard['net_revenue'].quantile(0.65)
product_scorecard['sales_strength'] = product_scorecard['net_revenue'] / product_scorecard['net_revenue'].max()
product_scorecard['rating_risk'] = ((5 - product_scorecard['average_rating']) / 4).clip(0, 1)
product_scorecard['attention_score'] = (
    product_scorecard['sales_strength']
    * (0.60 * product_scorecard['rating_risk'] + 0.40 * product_scorecard['low_rating_share'])
)
product_scorecard['candidate_for_review'] = (
    (product_scorecard['review_count'] >= 20)
    & (product_scorecard['net_revenue'] >= sales_cutoff)
    & (
        (product_scorecard['average_rating'] <= 3.3)
        | (product_scorecard['low_rating_share'] >= 0.20)
    )
)

product_scorecard = product_scorecard.sort_values(
    ['candidate_for_review', 'attention_score'], ascending=[False, False]
).reset_index(drop=True)
review_candidates = product_scorecard[product_scorecard['candidate_for_review']].copy()

scorecard_path = OUT_DIR / 'product_feedback_purchase_scorecard.csv'
candidates_path = OUT_DIR / 'product_investigation_candidates.csv'
product_scorecard.to_csv(scorecard_path, index=False)
review_candidates.to_csv(candidates_path, index=False)

show_table(product_scorecard, rows=20)
print('Sales cutoff used:', round(sales_cutoff, 2))
print('Candidate count:', len(review_candidates))

plt.figure(figsize=(8.0, 5.0))
non_candidates = product_scorecard[~product_scorecard['candidate_for_review']]
plt.scatter(non_candidates['net_revenue'], non_candidates['average_rating'], s=70, alpha=0.75, label='Other products')
if not review_candidates.empty:
    plt.scatter(review_candidates['net_revenue'], review_candidates['average_rating'], s=110, alpha=0.85, label='Investigation candidates')
    for _, row in review_candidates.iterrows():
        plt.annotate(row['name'], (row['net_revenue'], row['average_rating']), xytext=(4, 4), textcoords='offset points', fontsize=8)
plt.axvline(sales_cutoff, linestyle='--', linewidth=1, label='Sales cutoff')
plt.axhline(3.3, linestyle=':', linewidth=1, label='Rating review line')
plt.xlabel('Net revenue through the as-of date')
plt.ylabel('Average rating')
plt.title('Strong sales and weak feedback require joint investigation')
plt.legend()
mini_case_chart_path = save_current_figure('sales_and_review_risk.png')

In [ ]:
# ============================================================
# D.11.4 Identify complaint themes and materialize a reporting collection
# ============================================================
candidate_product_ids = review_candidates['product_id'].tolist()

candidate_theme_pipeline = [
    {'$match': {
        'product_id': {'$in': candidate_product_ids},
        'rating': {'$lte': 2},
        'created_at': {'$lte': AS_OF_DATE},
    }},
    {'$unwind': '$aspects'},
    {'$group': {
        '_id': {'product_id': '$product_id', 'aspect': '$aspects'},
        'mention_count': {'$sum': 1},
    }},
    {'$sort': {'_id.product_id': 1, 'mention_count': -1}},
]

candidate_theme_rows = [
    {
        'product_id': row['_id']['product_id'],
        'aspect': row['_id']['aspect'],
        'mention_count': row['mention_count'],
    }
    for row in reviews.aggregate(candidate_theme_pipeline)
]
candidate_themes = pd.DataFrame(candidate_theme_rows)
if not candidate_themes.empty:
    candidate_themes = candidate_themes.merge(
        product_context[['product_id', 'name']], on='product_id', how='left'
    )

candidate_themes_path = OUT_DIR / 'candidate_complaint_themes.csv'
candidate_themes.to_csv(candidate_themes_path, index=False)
show_table(candidate_themes, rows=30)

reporting_collection = db['product_feedback_summary']
reporting_collection.delete_many({})
materialized_records = dataframe_records(product_scorecard)
for record in materialized_records:
    record['_id'] = record.pop('product_id')
    record['as_of_date'] = AS_OF_DATE
if materialized_records:
    reporting_collection.insert_many(materialized_records)

reporting_preview = cursor_df(
    reporting_collection.find(
        {'candidate_for_review': True},
        {'_id': 1, 'name': 1, 'net_revenue': 1, 'average_rating': 1, 'attention_score': 1, 'as_of_date': 1},
    ).sort('attention_score', DESCENDING)
)
show_table(reporting_preview, rows=20)

## D.12 Governance, reproducibility, and handoff

A responsible document-database workflow defines document grain, field meanings, allowed types, reference checks, index intent, time boundaries, access privileges, and retention rules. It minimizes the data moved into notebooks, keeps credentials out of files, and saves both the aggregation logic and the resulting decision dataset. The in-memory environment used here is an instructional substitute, not evidence of production performance or security.

In [ ]:
# ============================================================
# D.12.1 Responsible NoSQL checklist
# ============================================================
responsible_nosql_checklist = pd.DataFrame([
    {'item': 'The business question and document grain are stated', 'status': 'complete'},
    {'item': 'Embedding and referencing choices follow access patterns', 'status': 'complete'},
    {'item': 'Critical field names, types, and allowed values are documented', 'status': 'complete'},
    {'item': 'Optional fields are distinguished from required fields', 'status': 'complete'},
    {'item': 'Referenced business identifiers are audited', 'status': 'complete'},
    {'item': 'Embedded arrays are bounded and document size is monitored', 'status': 'complete'},
    {'item': 'Indexes correspond to repeated filters and sorts', 'status': 'complete'},
    {'item': 'Database filtering and projection precede pandas conversion', 'status': 'complete'},
    {'item': 'As-of dates prevent future information from entering metrics', 'status': 'complete'},
    {'item': 'Credentials are stored outside shared notebooks', 'status': 'complete'},
    {'item': 'Production users receive least-privilege access', 'status': 'verify in deployment'},
    {'item': 'Retention, deletion, and sensitive-data policies are documented', 'status': 'verify with real data'},
    {'item': 'Notebook runs from a clean runtime', 'status': 'verify before sharing'},
])

platform_limitations = pd.DataFrame([
    {
        'environment': 'mongomock in this notebook',
        'appropriate_use': 'Syntax practice, document inspection, and small analytical demonstrations',
        'do_not_infer': 'Production latency, query plans, authentication, sharding, durability, or transaction behavior',
    },
    {
        'environment': 'MongoDB Atlas or managed deployment',
        'appropriate_use': 'Collaborative and production-oriented database workflows with configured security',
        'do_not_infer': 'That default settings automatically satisfy every governance requirement',
    },
])

show_table(responsible_nosql_checklist, rows=20)
show_table(platform_limitations, rows=10)

In [ ]:
# ============================================================
# D.12.2 Save pipelines, collection exports, and a handoff card
# ============================================================
review_pipeline_path = OUT_DIR / 'review_summary_pipeline.json'
transaction_pipeline_path = OUT_DIR / 'transaction_summary_pipeline.json'
decision_contract_path = OUT_DIR / 'product_investigation_decision_contract.json'

save_json(review_summary_pipeline, review_pipeline_path)
save_json(transaction_summary_pipeline, transaction_pipeline_path)
save_json({
    'as_of_date': AS_OF_DATE,
    'minimum_review_count': 20,
    'sales_cutoff_rule': '65th percentile of product net revenue in this synthetic dataset',
    'sales_cutoff_value': sales_cutoff,
    'quality_rule': 'average_rating <= 3.3 OR low_rating_share >= 0.20',
    'attention_score': 'sales_strength * (0.60 * rating_risk + 0.40 * low_rating_share)',
    'managerial_use': 'Prioritize human investigation; do not automate product removal',
}, decision_contract_path)

products_export_path = DATA_DIR / 'products_collection_export.jsonl'
reviews_export_path = DATA_DIR / 'reviews_collection_export.jsonl'
transactions_export_path = DATA_DIR / 'transactions_collection_export.jsonl'
save_jsonl(products.find(), products_export_path)
save_jsonl(reviews.find(), reviews_export_path)
save_jsonl(transactions.find(), transactions_export_path)

collection_counts = {
    collection_name: db[collection_name].count_documents({})
    for collection_name in sorted(db.list_collection_names())
}

handoff_card = {
    'notebook_name': 'Appendix_D_NoSQL_and_Document_Oriented_Databases.ipynb',
    'purpose': 'Beginner practice with document modeling, PyMongo-style operations, aggregation, quality checks, and document-to-Python analysis',
    'execution_environment': 'mongomock in-memory MongoDB-compatible interface',
    'production_transfer_target': 'MongoDB or MongoDB Atlas through PyMongo',
    'data_source': 'Synthetic e-commerce products, reviews, and transaction documents generated inside the notebook',
    'random_seed': SEED,
    'database_name': db.name,
    'collection_counts': collection_counts,
    'collection_grains': {
        'products': 'one product',
        'reviews': 'one customer review',
        'transactions': 'one transaction with a bounded line-item array',
        'product_feedback_summary': 'one product as of a stated cutoff',
    },
    'as_of_date': AS_OF_DATE,
    'indexes': {
        'reviews': reviews.index_information(),
        'transactions': transactions.index_information(),
    },
    'primary_outputs': [
        focused_reviews_path.name,
        source_summary_path.name,
        source_chart_path.name,
        schema_path.name,
        validation_report_path.name,
        scorecard_path.name,
        candidates_path.name,
        candidate_themes_path.name,
        mini_case_chart_path.name,
        review_pipeline_path.name,
        transaction_pipeline_path.name,
        decision_contract_path.name,
        document_size_chart_path.name,
    ],
    'known_limitations': [
        'The data are synthetic and do not represent a real retailer.',
        'mongomock does not reproduce production performance, query planning, networking, authentication, sharding, durability, or all MongoDB features.',
        'The prioritization score is descriptive and does not estimate causal impact.',
        'The mini-case omits margins, inventory constraints, return rates, and intervention costs.',
        'The JSON Schema example allows additional properties so that optional fields remain flexible.',
    ],
    'created_utc': datetime.now(timezone.utc),
}

handoff_path = OUT_DIR / 'nosql_notebook_handoff_card.json'
save_json(handoff_card, handoff_path)

artifact_paths = sorted(DATA_DIR.glob('*')) + sorted(OUT_DIR.glob('*'))
artifact_manifest = pd.DataFrame([
    {
        'artifact': str(path.relative_to(WORK_DIR)),
        'size_bytes': path.stat().st_size,
        'sha256_prefix': sha256_file(path)[:16],
    }
    for path in artifact_paths
    if path.is_file()
])

show_table(artifact_manifest, rows=40)
print(f'Handoff card saved to: {handoff_path}')

## Decision guide

Use a document database when records are naturally nested, optional fields are meaningful, the structure is expected to evolve, or related context is usually retrieved together. Use embedding for small, bounded child data that belong to one parent. Use referencing when related records grow, are reused, change independently, or require their own analytics. Define indexes from repeated access patterns rather than indexing every field. Filter, project, and aggregate in the database before moving results into pandas. Use a relational database when strict referential integrity, highly structured transactions, and complex repeatable joins dominate the problem. In many organizations, relational and document databases coexist.

## Exercises

1. Add a fifth business scenario to `business_choice_examples` and justify the most natural database model.

2. Modify `sample_review_document` by adding a nested delivery object. Flatten the result with `pd.json_normalize` and inspect the new column names.

3. Add one product whose `attributes` differ from every existing category. Insert it into `products` and verify that the collection accepts the new shape.

4. Query reviews from Gold customers with ratings at or below 2. Project only the product ID, rating, review text, region, and date.

5. Write a query that finds documents whose `aspects` array contains either `comfort` or `durability`. Sort the result by rating and date.

6. Insert a temporary review, update its source and helpful-vote count, verify the change, and delete the record.

7. Explain whether transaction line items should remain embedded. Discuss bounded growth, read patterns, and independent analysis.

8. Create an index for queries that filter by language and sort by created date. Add the index definition to the access-pattern guide.

9. Build an aggregation pipeline that reports review count and average rating by source.

10. Use `$unwind` to count complaint aspects for one selected category. Add product context with `$lookup` before grouping or filter by a list of product IDs.

11. Change the as-of date to April 30, 2026. Rebuild review and transaction metrics and explain why the candidate list changes.

12. Add `verified_purchase` to the review summary. Compare average rating and low-rating share for verified and unverified reviews.

13. Add another intentionally flawed record to `raw_reviews_quality_lab`. Rerun the field profile and schema validation report.

14. Write a cleanup function that maps `stars` to `rating`, `review_txt` to `review_text`, and valid date strings to ISO timestamps. Preserve the original raw record for lineage.

15. Add a minimum unique-customer rule to the mini-case. Save the revised decision contract and candidate file under new names.

16. Connect to MongoDB Atlas only through `MONGODB_URI`. Confirm that the connection cell does not print the URI or password.

17. Restart the runtime and run all cells. Correct any failure caused by hidden state rather than manually recreating missing variables.

In [ ]:
# Optional exercise starter: summarize review activity by source.
exercise_pipeline = [
    {'$match': {'created_at': {'$lte': AS_OF_DATE}}},
    {'$group': {
        '_id': '$source',
        'review_count': {'$sum': 1},
        'average_rating': {'$avg': '$rating'},
    }},
    {'$sort': {'review_count': -1}},
]

exercise_summary = cursor_df(reviews.aggregate(exercise_pipeline)).rename(columns={'_id': 'source'})
show_table(exercise_summary, rows=10)